### Train Insect Detector + Pollinator Classifier

Trains the two models used by the **two_stage** pipeline from scratch:
1. **Binary classifier** (`binary_best.pth`) — EfficientNet-B2, insect vs background
2. **Group classifier** (`4group_insectnet.pth`) — InsectNet, 4-class: bumblebee / fly / butterfly / other

**Input** — `data/training/annotated_crops/{bumblebee,fly,butterfly,other,background}/` (each folder must contain images)  
**Output** — `outputs/training/model_runs/{RUN_NAME}_{ts}/` (logs + curves) · **overwrites** `models/binary_best.pth` and `models/4group_insectnet.pth`

**Must edit (Cell 2):**

| Variable | Risk if skipped |
|----------|-----------------|
| `BINARY_BACKBONE` | `ValueError` if not `'insectnet'`, `'efficientnet'`, or `'both'` |

**Optional (Cell 2):** `EPOCHS_BINARY` / `EPOCHS_S1` (default 20), `EPOCHS_S2` (default 0 — skip Stage 2, recommended for InsectNet), `LR_S1` (1e-3), `LR_S2` (1e-4), `BG_RATIO` (3), `BATCH` (32), `USE_WEB_FOR_BINARY` (True), `DATASETS` ([] = all sub-folders)

**Background sampling:** balanced across camera plots — each plot contributes an equal quota so busy plots cannot dominate the training set. Both filename formats are handled automatically:
- **Current:** `Site_species_plot_date__Camera__Image_crop.jpg`
- **Legacy HDD:** `hdd_N_year_site_species_plot[_date]_NNN_WSCT__Image_crop.jpg` — camera-overflow folders (`_101_WSCT`, `_102_WSCT`, …) are the same physical camera hitting the 9 999-image folder limit and are merged into one plot key automatically.

If the filename format changes in a future season, update only `parse_plot_key()` — see its docstring.

Quota is based on **field insect crop count only** (web images do not inflate the background target).


##### Cell 1 — Environment  *(no edits needed)*

**Local:** auto-detects the repo root via `git rev-parse --show-toplevel` — no path editing required.

**Colab:** uses the path extracted from the zip in Cell 0.

Sets all derived paths (`MODEL_DIR`, `LABELED_DIR`, output folders).

In [ ]:
import os
# Fix macOS OpenMP duplicate-library crash (libomp.dylib loaded twice → SIGABRT)
# Must be set before ANY torch/torchvision import.
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '1')

import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    import zipfile, os
    DRIVE_ROOT = Path('/content/drive/MyDrive')
    ZIP_PATH   = DRIVE_ROOT / 'pollinator-colab.zip'
    EXTRACT_TO = Path('/content/pollinator-colab')
    if not EXTRACT_TO.exists():
        print(f'Extracting {ZIP_PATH.name} ...')
        with zipfile.ZipFile(ZIP_PATH) as z:
            z.extractall('/content/')
        print('✓ Extracted to /content/pollinator-colab')
    else:
        print('✓ Already extracted')
    BASE_DIR   = EXTRACT_TO
    DRIVE_BASE = DRIVE_ROOT / 'pollinator-colab'
else:
    import subprocess as _sp
    _git_root  = Path(_sp.check_output(
        ['git', 'rev-parse', '--show-toplevel'], text=True).strip())
    BASE_DIR   = _git_root / 'ml_pipelines' / 'notebooks' / 'pollinator_detection'
    DRIVE_BASE = BASE_DIR

MODEL_DIR   = BASE_DIR / 'models'
LABELED_DIR = BASE_DIR / 'data' / 'training' / 'annotated_crops'
INSECTNET_W = BASE_DIR / 'InsectNet' / 'model.pth'
WEB_IMG_DIR = BASE_DIR / 'data' / 'web_images'

# Training outputs go to local SSD on Colab (fast); saved to Drive after training.
LOCAL_TRAINING = Path('/content/outputs/training') if IN_COLAB else BASE_DIR / 'outputs' / 'training'
LOCAL_TRAINING.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f'Env        : {"Colab" if IN_COLAB else "Local"}')
print(f'BASE_DIR   : {BASE_DIR}  exists={BASE_DIR.exists()}')
print(f'MODEL_DIR  : {MODEL_DIR}  exists={MODEL_DIR.exists()}')
print(f'LABELED_DIR: {LABELED_DIR}  exists={LABELED_DIR.exists()}')
print(f'INSECTNET_W: {INSECTNET_W}  exists={INSECTNET_W.exists()}')
print(f'WEB_IMG_DIR: {WEB_IMG_DIR}  exists={WEB_IMG_DIR.exists()}')


##### Cell 2 — Config  ← **edit before training**
Set backbone, epochs, LR, and batch size.

In [ ]:
if 'BASE_DIR' not in dir():
    raise RuntimeError(
        '\u26a0\ufe0f  BASE_DIR is not defined. '
        'Please run Cell 1 (environment setup) first.'
    )

# ── Which sub-datasets to use ──────────────────────────────────────
# Leave [] to use ALL sub-folders automatically (excluding 'progress').
# Or list specific ones: ['labeled_ls', 'labeled_mb']
DATASETS = []

RUN_NAME     = 'binary_group'   # ← label appended to timestamp

CLASSES_4      = ['bumblebee','fly','butterfly','other']
CLASSES_BINARY = ['background','insect']
INSECT_FOLDERS = ['bumblebee','fly','butterfly','other']

# Folder name -> canonical class  (handles legacy naming)
ALIAS_4 = {
    'bumblebee':'bumblebee', 'fly':'fly',
    'butterfly':'butterfly', 'butterfly_moth':'butterfly',
    'other':'other',
}

IMG_SIZE       = 224
BATCH          = 32
BINARY_BACKBONE = 'both'  # 'efficientnet' | 'insectnet' | 'both'
EPOCHS_BINARY  = 20
EPOCHS_S1      = 20    # Stage 1: web + arctic combined
EPOCHS_S2      = 0     # Stage 2: arctic only (0 = skip, recommended for InsectNet)
LR_S1          = 1e-3
LR_S2          = 1e-4
BG_RATIO       = 4     # background:insect sampling ratio
SEED           = 42

WEB_DIR            = BASE_DIR / 'data' / 'web_images'  # iNaturalist images folder
USE_WEB_FOR_BINARY = True   # add web images as extra insect data for binary classifier
WEB_ALIAS_4 = {
    'bumblebee':'bumblebee','fly':'fly',
    'butterfly':'butterfly','butterfly_moth':'butterfly','other':'other',
}

print(f'LABELED_DIR: {LABELED_DIR}  exists={LABELED_DIR.exists()}')
print(f'INSECTNET  : {INSECTNET_W}  exists={INSECTNET_W.exists()}')


##### Cell 2b — Output paths
Builds the timestamped run directory. No edits needed.

In [ ]:
if 'BASE_DIR' not in dir():
    raise RuntimeError(
        '\u26a0\ufe0f  BASE_DIR is not defined. '
        'Please run Cell 1 (environment setup) first.'
    )

from datetime import datetime

_ts     = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = BASE_DIR / 'outputs' / 'training' / 'model_runs' / f'{RUN_NAME}_{_ts}'
RUN_DIR.mkdir(parents=True, exist_ok=True)

import json as _json
_json.dump({
    'run_name': RUN_NAME, 'timestamp': _ts,
    'epochs_binary': EPOCHS_BINARY, 'epochs_s1': EPOCHS_S1, 'epochs_s2': EPOCHS_S2,
    'lr_s1': LR_S1, 'lr_s2': LR_S2, 'img_size': IMG_SIZE, 'batch': BATCH,
    'classes_4': CLASSES_4, 'classes_binary': CLASSES_BINARY, 'bg_ratio': BG_RATIO,
}, open(RUN_DIR / 'config.json', 'w'), indent=2)

print(f'Run directory : {RUN_DIR}')
print('Checkpoints + curves saved here; best models also copied to models/')

# ── Resolve dataset sub-folders ─────────────────────────────────
if DATASETS:
    DATASET_DIRS = [LABELED_DIR / ds for ds in DATASETS]
else:
    DATASET_DIRS = sorted([d for d in LABELED_DIR.iterdir()
                           if d.is_dir() and d.name != 'progress'])
print(f'Datasets : {[d.name for d in DATASET_DIRS]}')

# ── Per-model output sub-folders ─────────────────────────────────
BINARY_DIR = RUN_DIR / 'binary'
GROUP_DIR  = RUN_DIR / 'group'
BINARY_DIR.mkdir(parents=True, exist_ok=True)
GROUP_DIR.mkdir(parents=True, exist_ok=True)
print(f'Binary dir: {BINARY_DIR}')
print(f'Group dir : {GROUP_DIR}')


##### Cell 3 — Imports + training utilities
Loads PyTorch, model definitions, and training helpers. Just run.

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
try:
    from sklearn.metrics import classification_report
    HAS_SKLEARN = True
except ImportError:
    HAS_SKLEARN = False
DEVICE = (torch.device('cuda') if torch.cuda.is_available()
             else torch.device('mps') if torch.backends.mps.is_available()
             else torch.device('cpu'))
print(f'Device: {DEVICE}  |  PyTorch: {torch.__version__}')


##### Cell 3b — Plotting utilities
Defines loss/accuracy curve helpers. Just run.

In [ ]:

def _sync_to_drive(run_dir, drive_base, label):

    """Copy run_dir (including all sub-folders) to Drive immediately."""
    if not IN_COLAB:
        return
    import shutil
    dest = Path(drive_base) / 'outputs' / 'training' / 'model_runs' / run_dir.name
    if dest.exists():
        shutil.rmtree(str(dest))
    shutil.copytree(str(run_dir), str(dest))
    n = sum(1 for f in dest.rglob('*') if f.is_file())
    print(f'  ✓ [{label}] synced {n} files → Drive: {dest}')

import shutil, matplotlib

matplotlib.use('Agg')

import matplotlib.pyplot as plt

from collections import defaultdict

import numpy as np, torch, torch.nn as nn

import torchvision, torchvision.transforms as T

from PIL import Image

from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

try:

    from sklearn.metrics import classification_report
    HAS_SKLEARN=True

except ImportError:

    HAS_SKLEARN=False

DEVICE = (torch.device('cuda') if torch.cuda.is_available()
          else torch.device('mps') if torch.backends.mps.is_available()
          else torch.device('cpu'))

print(f'Device: {DEVICE}')

def letterbox(img, size):

    w,h=img.size; ms=max(w,h)
    sq=Image.new('RGB',(ms,ms),(0,0,0)); sq.paste(img,((ms-w)//2,(ms-h)//2))
    return sq.resize((size,size),Image.BILINEAR)

class CropDataset(Dataset):

    def __init__(self, samples, tf): self.s=samples; self.tf=tf
    def __len__(self): return len(self.s)
    def __getitem__(self,i):
        p,l=self.s[i]; return self.tf(Image.open(p).convert('RGB')), l

class _Letterbox:

    """Picklable letterbox transform — needed for DataLoader num_workers > 0.
    A lambda inside T.Lambda cannot be pickled by multiprocessing on macOS.
    """
    def __init__(self, size): self.size = size
    def __call__(self, img): return letterbox(img, self.size)

def make_tf(sz, aug=False):

    base = [_Letterbox(sz), T.ToTensor(),
            T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]
    if aug:
        base = [
            _Letterbox(sz),
            T.RandomHorizontalFlip(),
            T.RandomVerticalFlip(),
            T.RandomRotation(degrees=30),                          # rotation — insects appear at any angle
            T.ColorJitter(brightness=0.4, contrast=0.4,            # stronger colour jitter
                          saturation=0.3, hue=0.08),
            T.RandomGrayscale(p=0.05),                             # occasional greyscale — extreme lighting
            T.RandomAffine(degrees=0, translate=(0.1, 0.1),        # slight translate + scale
                           scale=(0.85, 1.15)),
        ] + base[1:]
    return T.Compose(base)

def make_loader(samples, idxs, sz, batch, aug=False, weighted=True):

    sub=[samples[i] for i in idxs]; ds=CropDataset(sub,make_tf(sz,aug))
    if weighted and sub:
        labs=[s[1] for s in sub]; cnt=np.bincount(labs,minlength=max(labs)+1)
        wts=[1.0/max(1,cnt[l]) for l in labs]
        return DataLoader(ds,batch_size=batch,
                          sampler=WeightedRandomSampler(wts,len(wts)),num_workers=0,pin_memory=torch.cuda.is_available())
    return DataLoader(ds,batch_size=batch,shuffle=False,num_workers=0,pin_memory=torch.cuda.is_available())

def parse_plot_key(path):
    """
    Extract a per-camera-plot group key from a crop filename for stratified
    background sampling. Returns a string identifying one plot (site+species+plot+date).

    CURRENT FORMAT: <site>__<camera>__<image>_crop<i>.jpg
      e.g. Gruvan_Bal_p1_20250727__102_WSCT__WSCT1529_crop00.jpg
      Returns: Gruvan_Bal_p1_20250727

    LEGACY FORMAT (hdd_*):
      hdd_<n>_<year>_<site>_<species>_<plot>[_<date>]_<NNN>_WSCT__<image>.jpg
      The _101_WSCT/_102_WSCT/... suffix is the SAME camera overflowing at
      9999 images into a new folder. We strip this so all overflow folders
      share one plot key.
      e.g. hdd_2_2025_desert_Vau_p1_20250725_101_WSCT__...
           hdd_2_2025_desert_Vau_p1_20250725_102_WSCT__...
           both return: hdd_2_2025_desert_Vau_p1_20250725
    """
    import re as _re
    stem = Path(path).stem
    # Both formats use __ as separator — take everything before the first __
    base = stem.split('__')[0] if '__' in stem else stem
    # Strip accidental whitespace in folder names
    base = base.strip()
    # Strip trailing camera-overflow suffix _NNN_WSCT (case-insensitive)
    base = _re.sub(r'_\d+_[Ww][Ss][Cc][Tt]$', '', base)
    return base if base else 'unknown'

def sample_bg(bg_paths, n_total, seed=42):

    """
    Sample n_total background crops balanced across camera plots.
    Uses parse_plot_key() to group crops by plot, then gives each plot an
    equal quota so that no single busy plot can dominate the training set.
    Args:
        bg_paths : list of Path — all available background crop paths
        n_total  : int — how many to sample in total
        seed     : int — RNG seed for reproducibility
    Prints a per-group breakdown so you can verify the balance.
    If a plot has fewer crops than its quota, it contributes all it has
    (the total sampled may be slightly below n_total in that case).
    """
    if not n_total:
        return []
    groups = {}
    for p in bg_paths:
        groups.setdefault(parse_plot_key(p), []).append(p)
    rng = np.random.default_rng(seed)
    for imgs in groups.values():
        rng.shuffle(imgs)
    n_groups  = len(groups)
    quota     = n_total // n_groups
    remainder = n_total  % n_groups
    print(f'Background sampling: {n_groups} plot group(s), quota={quota}/group')
    for key, imgs in sorted(groups.items()):
        print(f'  {key:30}: {len(imgs):>5} available')
    sampled = []
    for i, key in enumerate(sorted(groups)):
        take = quota + (1 if i < remainder else 0)
        sampled.extend(groups[key][:min(take, len(groups[key]))])
    rng.shuffle(sampled)
    print(f'Sampled {len(sampled)} background crops (target {n_total})')
    return sampled

def stratified_split(samples, val_frac=0.1, test_frac=0.1, seed=42):

    """Split list of (path, label) into train/val/test indices, stratified by class."""
    rng = np.random.default_rng(seed)
    by_cls = defaultdict(list)
    for i, (_, l) in enumerate(samples): by_cls[l].append(i)
    tr, va, te = [], [], []
    for l, idxs in by_cls.items():
        rng.shuffle(idxs); n = len(idxs)
        nva = max(1, int(n*val_frac)); nte = max(1, int(n*test_frac))
        tr.extend(idxs[nva+nte:]); va.extend(idxs[:nva]); te.extend(idxs[nva:nva+nte])
    return tr, va, te

def collect_crops(dataset_dirs, classes, alias):

    """Collect (path, label_idx) tuples from sub-folders in each dataset dir."""
    ci = {c: i for i, c in enumerate(classes)}
    smp = []
    for labeled_dir in dataset_dirs:
        for folder, cls in alias.items():
            if cls not in ci:
                continue
            dd = Path(labeled_dir) / folder
            if not dd.exists():
                continue
            for ext in ('*.jpg', '*.jpeg', '*.png'):
                for p in dd.glob(ext):
                    smp.append((p, ci[cls]))
    counts = {i: 0 for i in range(len(classes))}
    for _, l in smp:
        counts[l] += 1
    return smp, counts

def build_efficientnet(n_classes):

    """EfficientNet-B2 pretrained on ImageNet, head replaced for n_classes."""
    print(f'Building EfficientNet-B2 (ImageNet weights, {n_classes} classes)')
    model = torchvision.models.efficientnet_b2(weights='IMAGENET1K_V1')
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, n_classes)
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  {n_train:,} trainable params (full fine-tune from ImageNet)')
    return model

def build_insectnet(weights_path, n_classes, unfreeze_last_block=False):

    """InsectNet backbone (RegNet-Y-32GF), frozen by default, head replaced for n_classes."""
    print(f'Building InsectNet ({n_classes} classes, RegNet-Y-32GF backbone)')
    if not Path(weights_path).exists():
        raise FileNotFoundError(f'InsectNet weights not found: {weights_path}')
    print(f'  Loading weights: {weights_path}')
    model = torchvision.models.regnet_y_32gf()
    model.fc = nn.Linear(3712, 2526)
    state = torch.load(weights_path, map_location='cpu', weights_only=False)
    model.load_state_dict(state['model'] if 'model' in state else state, strict=True)
    print('  InsectNet weights loaded ✓')
    model.fc = nn.Linear(3712, n_classes)
    nn.init.xavier_uniform_(model.fc.weight)
    nn.init.zeros_(model.fc.bias)
    for name, p in model.named_parameters():
        p.requires_grad = name.startswith('fc.')
    if unfreeze_last_block:
        for name, p in model.named_parameters():
            if 'trunk_output.block4' in name or name.startswith('fc.'):
                p.requires_grad = True
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    frozen_str = 'frozen backbone' if not unfreeze_last_block else 'last block + fc unfrozen'
    print(f'  Trainable: {n_train:,} / {n_total:,} params ({frozen_str})')
    return model

def _train_epoch(model, loader, optimizer, criterion, device):

    """Single training epoch; returns (avg_loss, accuracy)."""
    model.train()
    ls = cor = tot = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        preds = out.argmax(1)
        ls  += loss.item() * labels.size(0)
        cor += (preds == labels).sum().item()
        tot += labels.size(0)
    return ls / tot, cor / tot

@torch.no_grad()

def eval_epoch(model, loader, criterion, device, classes):

    """Evaluate model; returns dict with loss, acc, macro_f1, per_f1, preds, labels."""
    model.eval()
    ls = cor = tot = 0
    ap, al = [], []
    tp = {i: 0 for i in range(len(classes))}
    fp = {i: 0 for i in range(len(classes))}
    fn = {i: 0 for i in range(len(classes))}
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out = model(imgs)
        preds = out.argmax(1)
        ls  += criterion(out, labels).item() * labels.size(0)
        cor += (preds == labels).sum().item()
        tot += labels.size(0)
        ap.extend(preds.cpu().tolist())
        al.extend(labels.cpu().tolist())
        for i in range(len(classes)):
            tp[i] += ((preds == i) & (labels == i)).sum().item()
            fp[i] += ((preds == i) & (labels != i)).sum().item()
            fn[i] += ((preds != i) & (labels == i)).sum().item()
    pf = {}; pr = {}
    for i in range(len(classes)):
        p2 = tp[i] / max(1, tp[i] + fp[i])
        r  = tp[i] / max(1, tp[i] + fn[i])
        pf[classes[i]] = 2 * p2 * r / max(1e-8, p2 + r)
        pr[classes[i]] = r
    # Combined recall across all non-background classes:
    # of all actual insects, what fraction was NOT missed as background?
    _ins_cls = [c for c in classes if c != 'background']
    _ins_idx = [i for i, c in enumerate(classes) if c != 'background']
    _ins_tp  = sum(tp[i] for i in _ins_idx)
    _ins_fn  = sum(fn[i] for i in _ins_idx)
    insect_recall = _ins_tp / max(1, _ins_tp + _ins_fn)
    # per-class precision (needed for webapp-compatible results JSON)
    pp = {}
    for i in range(len(classes)):
        _p = tp[i] / max(1, tp[i] + fp[i])
        pp[classes[i]] = _p
    return {'loss': ls / max(1, tot), 'acc': cor / max(1, tot),
            'macro_f1': sum(pf.values()) / max(1, len(classes)),
            'per_f1': pf, 'per_recall': pr, 'per_precision': pp,
            'per_tp': {classes[i]: tp[i] for i in range(len(classes))},
            'per_fp': {classes[i]: fp[i] for i in range(len(classes))},
            'per_fn': {classes[i]: fn[i] for i in range(len(classes))},
            'insect_recall': insect_recall,
            'preds': ap, 'labels': al}

def run_training(model, name, tr_ldr, va_ldr, epochs, lr, counts, ckpt,

                 device, classes, run_dir, in_colab,
                 best_metric='macro_f1'):
    """Full training loop with cosine LR schedule.
    Saves the best checkpoint by val metric (default macro_f1).
    For binary insect detection set best_metric='recall_insect' to avoid
    missing insects — recall of the insect class is printed each epoch.
    Returns the model loaded with the best weights.
    """
    w = torch.tensor([1.0 / max(1, counts.get(i, 1)) for i in range(len(classes))],
                     dtype=torch.float, device=device)
    crit  = nn.CrossEntropyLoss(weight=w / w.sum())
    opt   = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best_f1 = 0.0
    hist    = {'tr': [], 'va': [], 'f1': []}
    hdr = (f'{"Ep":>4}  {"TrLoss":>8}  {"VaLoss":>8}  {"MacroF1":>8}'
           f'  {"InsRec":>8}  {"Acc":>6}  '
           + '  '.join(f'{c[:8]:>9}' for c in classes))
    print(f'\n{"=" * 70}\n{name}  epochs={epochs}  lr={lr}  best={best_metric}\n{"=" * 70}\n{hdr}')
    for ep in range(1, epochs + 1):
        tl, ta = _train_epoch(model, tr_ldr, opt, crit, device)
        vr     = eval_epoch(model, va_ldr, crit, device, classes)
        sched.step()
        # Select best checkpoint by the chosen metric:
        #   'macro_f1'      — balanced across all classes (default, group classifier)
        #   'recall_insect' — insect-class recall (binary: don't miss insects)
        #   'recall_nonbg'  — combined recall of all non-background classes
        #                     (5-class model: don't call any real insect 'background')
        if best_metric == 'macro_f1':
            _score = vr['macro_f1']
        elif best_metric == 'recall_nonbg':
            _score = vr['insect_recall']
        else:
            # 'recall_<classname>'  e.g. 'recall_insect'
            _cls = best_metric.split('_', 1)[1]
            _score = vr['per_recall'].get(_cls, vr['macro_f1'])
        new_best = _score > best_f1
        if new_best:
            best_f1 = _score
            torch.save({'state_dict': model.state_dict(), 'classes': classes,
                        'val_macro_f1': vr['macro_f1'], 'val_score': best_f1,
                        'best_val_f1': best_f1,  # webapp upload key
                        'val_acc': vr['acc'],
                        'val_per_class_f1': vr['per_f1'],
                        'best_metric': best_metric, 'epoch': ep}, ckpt)
        pf = vr['per_f1']
        if best_metric == 'macro_f1':
            _mscore = vr['macro_f1']
        elif best_metric == 'recall_nonbg':
            _mscore = vr['insect_recall']
        else:
            _cls = best_metric.split('_', 1)[1]
            _mscore = vr['per_recall'].get(_cls, 0)
        print(f'{ep:>4}  {tl:>8.4f}  {vr["loss"]:>8.4f}  '
              f'{vr["macro_f1"]:>8.3f}  {_mscore:>8.3f}  {vr["acc"]:>6.3f}  '
              + '  '.join(f'{pf.get(c, 0):>9.3f}' for c in classes)
              + ('  *' if new_best else ''))
        hist['tr'].append(tl)
        hist['va'].append(vr['loss'])
        hist['f1'].append(vr['macro_f1'])
    print(f'\nBest val {best_metric}: {best_f1:.3f}  →  {ckpt}')
    # Plot training curves
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(hist['tr'], label='train'); ax1.plot(hist['va'], label='val')
    ax1.set_title('Loss'); ax1.legend(); ax1.grid(True)
    ax2.plot(hist['f1'], lw=2, label='macro F1')
    ax2.set_title('Macro F1'); ax2.grid(True)
    plt.suptitle(name); plt.tight_layout()
    curves_path = Path(run_dir) / f'{name}_curves.png'
    plt.savefig(curves_path, dpi=100); plt.close()
    print(f'Curves: {curves_path}')
    # Load best weights before returning
    model.load_state_dict(
        torch.load(ckpt, map_location=device, weights_only=False)['state_dict'])
    return model

def collect_crops(dataset_dirs, classes, alias):

    """Collect (path, label_idx) tuples from sub-folders in each dataset dir."""
    ci = {c: i for i, c in enumerate(classes)}
    smp = []
    for labeled_dir in dataset_dirs:
        for folder, cls in alias.items():
            if cls not in ci:
                continue
            dd = Path(labeled_dir) / folder
            if not dd.exists():
                continue
            for ext in ('*.jpg', '*.jpeg', '*.png'):
                for p in dd.glob(ext):
                    smp.append((p, ci[cls]))
    counts = {i: 0 for i in range(len(classes))}
    for _, l in smp:
        counts[l] += 1
    return smp, counts

def build_efficientnet(n_classes):

    """EfficientNet-B2 pretrained on ImageNet, head replaced for n_classes."""
    print(f'Building EfficientNet-B2 (ImageNet weights, {n_classes} classes)')
    model = torchvision.models.efficientnet_b2(weights='IMAGENET1K_V1')
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, n_classes)
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  {n_train:,} trainable params (full fine-tune from ImageNet)')
    return model

def build_insectnet(weights_path, n_classes, unfreeze_last_block=False):

    """InsectNet backbone (RegNet-Y-32GF), frozen by default, head replaced for n_classes."""
    print(f'Building InsectNet ({n_classes} classes, RegNet-Y-32GF backbone)')
    if not Path(weights_path).exists():
        raise FileNotFoundError(f'InsectNet weights not found: {weights_path}')
    print(f'  Loading weights: {weights_path}')
    model = torchvision.models.regnet_y_32gf()
    model.fc = nn.Linear(3712, 2526)
    state = torch.load(weights_path, map_location='cpu', weights_only=False)
    model.load_state_dict(state['model'] if 'model' in state else state, strict=True)
    print('  InsectNet weights loaded ✓')
    model.fc = nn.Linear(3712, n_classes)
    nn.init.xavier_uniform_(model.fc.weight)
    nn.init.zeros_(model.fc.bias)
    for name, p in model.named_parameters():
        p.requires_grad = name.startswith('fc.')
    if unfreeze_last_block:
        for name, p in model.named_parameters():
            if 'trunk_output.block4' in name or name.startswith('fc.'):
                p.requires_grad = True
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    frozen_str = 'frozen backbone' if not unfreeze_last_block else 'last block + fc unfrozen'
    print(f'  Trainable: {n_train:,} / {n_total:,} params ({frozen_str})')
    return model

def _train_epoch(model, loader, optimizer, criterion, device):

    """Single training epoch; returns (avg_loss, accuracy)."""
    model.train()
    ls = cor = tot = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        preds = out.argmax(1)
        ls  += loss.item() * labels.size(0)
        cor += (preds == labels).sum().item()
        tot += labels.size(0)
    return ls / tot, cor / tot

@torch.no_grad()

def eval_epoch(model, loader, criterion, device, classes):

    """Evaluate model; returns dict with loss, acc, macro_f1, per_f1, preds, labels."""
    model.eval()
    ls = cor = tot = 0
    ap, al = [], []
    tp = {i: 0 for i in range(len(classes))}
    fp = {i: 0 for i in range(len(classes))}
    fn = {i: 0 for i in range(len(classes))}
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out = model(imgs)
        preds = out.argmax(1)
        ls  += criterion(out, labels).item() * labels.size(0)
        cor += (preds == labels).sum().item()
        tot += labels.size(0)
        ap.extend(preds.cpu().tolist())
        al.extend(labels.cpu().tolist())
        for i in range(len(classes)):
            tp[i] += ((preds == i) & (labels == i)).sum().item()
            fp[i] += ((preds == i) & (labels != i)).sum().item()
            fn[i] += ((preds != i) & (labels == i)).sum().item()
    pf = {}; pr = {}
    for i in range(len(classes)):
        p2 = tp[i] / max(1, tp[i] + fp[i])
        r  = tp[i] / max(1, tp[i] + fn[i])
        pf[classes[i]] = 2 * p2 * r / max(1e-8, p2 + r)
        pr[classes[i]] = r
    # Combined recall across all non-background classes:
    # of all actual insects, what fraction was NOT missed as background?
    _ins_cls = [c for c in classes if c != 'background']
    _ins_idx = [i for i, c in enumerate(classes) if c != 'background']
    _ins_tp  = sum(tp[i] for i in _ins_idx)
    _ins_fn  = sum(fn[i] for i in _ins_idx)
    insect_recall = _ins_tp / max(1, _ins_tp + _ins_fn)
    # per-class precision (needed for webapp-compatible results JSON)
    pp = {}
    for i in range(len(classes)):
        _p = tp[i] / max(1, tp[i] + fp[i])
        pp[classes[i]] = _p
    return {'loss': ls / max(1, tot), 'acc': cor / max(1, tot),
            'macro_f1': sum(pf.values()) / max(1, len(classes)),
            'per_f1': pf, 'per_recall': pr, 'per_precision': pp,
            'per_tp': {classes[i]: tp[i] for i in range(len(classes))},
            'per_fp': {classes[i]: fp[i] for i in range(len(classes))},
            'per_fn': {classes[i]: fn[i] for i in range(len(classes))},
            'insect_recall': insect_recall,
            'preds': ap, 'labels': al}

def run_training(model, name, tr_ldr, va_ldr, epochs, lr, counts, ckpt,

                 device, classes, run_dir, in_colab,
                 best_metric='macro_f1'):
    """Full training loop with cosine LR schedule.
    Saves the best checkpoint by val metric (default macro_f1).
    For binary insect detection set best_metric='recall_insect' to avoid
    missing insects — recall of the insect class is printed each epoch.
    Returns the model loaded with the best weights.
    """
    w = torch.tensor([1.0 / max(1, counts.get(i, 1)) for i in range(len(classes))],
                     dtype=torch.float, device=device)
    crit  = nn.CrossEntropyLoss(weight=w / w.sum())
    opt   = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    best_f1 = 0.0
    hist    = {'tr': [], 'va': [], 'f1': []}
    hdr = (f'{"Ep":>4}  {"TrLoss":>8}  {"VaLoss":>8}  {"MacroF1":>8}'
           f'  {"InsRec":>8}  {"Acc":>6}  '
           + '  '.join(f'{c[:8]:>9}' for c in classes))
    print(f'\n{"=" * 70}\n{name}  epochs={epochs}  lr={lr}  best={best_metric}\n{"=" * 70}\n{hdr}')
    for ep in range(1, epochs + 1):
        tl, ta = _train_epoch(model, tr_ldr, opt, crit, device)
        vr     = eval_epoch(model, va_ldr, crit, device, classes)
        sched.step()
        # Select best checkpoint by the chosen metric:
        #   'macro_f1'      — balanced across all classes (default, group classifier)
        #   'recall_insect' — insect-class recall (binary: don't miss insects)
        #   'recall_nonbg'  — combined recall of all non-background classes
        #                     (5-class model: don't call any real insect 'background')
        if best_metric == 'macro_f1':
            _score = vr['macro_f1']
        elif best_metric == 'recall_nonbg':
            _score = vr['insect_recall']
        else:
            # 'recall_<classname>'  e.g. 'recall_insect'
            _cls = best_metric.split('_', 1)[1]
            _score = vr['per_recall'].get(_cls, vr['macro_f1'])
        new_best = _score > best_f1
        if new_best:
            best_f1 = _score
            torch.save({'state_dict': model.state_dict(), 'classes': classes,
                        'val_macro_f1': vr['macro_f1'], 'val_score': best_f1,
                        'best_val_f1': best_f1,  # webapp upload key
                        'val_acc': vr['acc'],
                        'val_per_class_f1': vr['per_f1'],
                        'best_metric': best_metric, 'epoch': ep}, ckpt)
        pf = vr['per_f1']
        if best_metric == 'macro_f1':
            _mscore = vr['macro_f1']
        elif best_metric == 'recall_nonbg':
            _mscore = vr['insect_recall']
        else:
            _cls = best_metric.split('_', 1)[1]
            _mscore = vr['per_recall'].get(_cls, 0)
        print(f'{ep:>4}  {tl:>8.4f}  {vr["loss"]:>8.4f}  '
              f'{vr["macro_f1"]:>8.3f}  {_mscore:>8.3f}  {vr["acc"]:>6.3f}  '
              + '  '.join(f'{pf.get(c, 0):>9.3f}' for c in classes)
              + ('  *' if new_best else ''))
        hist['tr'].append(tl)
        hist['va'].append(vr['loss'])
        hist['f1'].append(vr['macro_f1'])
    print(f'\nBest val {best_metric}: {best_f1:.3f}  →  {ckpt}')
    # Plot training curves
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(hist['tr'], label='train'); ax1.plot(hist['va'], label='val')
    ax1.set_title('Loss'); ax1.legend(); ax1.grid(True)
    ax2.plot(hist['f1'], lw=2, label='macro F1')
    ax2.set_title('Macro F1'); ax2.grid(True)
    plt.suptitle(name); plt.tight_layout()
    curves_path = Path(run_dir) / f'{name}_curves.png'
    plt.savefig(curves_path, dpi=100); plt.close()
    print(f'Curves: {curves_path}')
    # Load best weights before returning
    model.load_state_dict(
        torch.load(ckpt, map_location=device, weights_only=False)['state_dict'])
    return model

print('Training utilities loaded.')


##### Cell 4 — Collect and split data

Loads annotated crops from all `DATASET_DIRS` and builds stratified train/val/test splits.
Two code cells: the first collects binary data (insect vs background), the second collects 4-class group data.

**Background sampling** is balanced across camera plots using `parse_plot_key()` + `sample_bg()`:
each plot contributes an equal quota so a busy plot cannot dominate the training set.
The bg quota is based on **field insect crop count only** (not web images).

> **If your crop filenames change** (new field season, different camera naming), update
> `parse_plot_key()` — that is the only function you need to touch.
>
> Supported formats out of the box:
> - `Site_species_plot_date__Camera__Image_crop.jpg` (current)
> - `hdd_N_year_site_species_plot[_date]_NNN_WSCT__Image_crop.jpg` (legacy HDD)
>   Camera-overflow folders (`_101_WSCT`, `_102_WSCT`, …) are automatically
>   merged — they share the same plot quota.


In [ ]:
from pathlib import Path

# ── Binary data ──────────────────────────────────────────────────
insect_paths = []
bg_paths_all = []
for ds_dir in DATASET_DIRS:
    for folder in INSECT_FOLDERS:
        for ext in ('*.jpg','*.jpeg','*.png'): insect_paths.extend((ds_dir/folder).glob(ext))
    for ext in ('*.jpg','*.jpeg','*.png'): bg_paths_all.extend((ds_dir/'background').glob(ext))

_field_insect_n = len(insect_paths)  # field crops only — used for bg quota

# ── Add web images to binary insect class ────────────────────────
if USE_WEB_FOR_BINARY and WEB_DIR and Path(WEB_DIR).exists():
    _web_insect = []
    for _folder in INSECT_FOLDERS:
        for _ext in ('*.jpg','*.jpeg','*.png'):
            _web_insect.extend(Path(WEB_DIR).glob(f'{_folder}/{_ext}'))
    insect_paths.extend(_web_insect)
    print(f'Web images added to insect class: {len(_web_insect)}')
else:
    print('USE_WEB_FOR_BINARY=False or WEB_DIR not found — binary uses field crops only')

# bg quota based on field crops only — web images inflate insect count
# but bg crops come from field data, so ratio should match field distribution
bg_sampled  = sample_bg(bg_paths_all, _field_insect_n * BG_RATIO, SEED)
binary_data = [(p,1) for p in insect_paths] + [(p,0) for p in bg_sampled]
bin_counts  = {0:len(bg_sampled), 1:len(insect_paths)}
bin_tr, bin_va, bin_te = stratified_split(binary_data, seed=SEED)
print(f'Binary: insect={bin_counts[1]}  bg={bin_counts[0]}')
print(f'Split : train={len(bin_tr)}  val={len(bin_va)}  test={len(bin_te)}')

# ── Group data ───────────────────────────────────────────────────
arctic_smp, counts_arc = collect_crops(DATASET_DIRS, CLASSES_4, ALIAS_4)
web_smp, counts_web = (collect_crops([WEB_DIR], CLASSES_4, WEB_ALIAS_4)
                       if WEB_DIR and Path(WEB_DIR).exists() else ([],{i:0 for i in range(4)}))
arc_tr, arc_va, arc_te = stratified_split(arctic_smp, seed=SEED)
web_tr, web_va, web_te = stratified_split(web_smp,    seed=SEED) if web_smp else ([],[],[])
print(f'\nGroup: arctic={len(arctic_smp)}  web={len(web_smp)}')
# ── Crop + web image count summary ───────────────────────────────
_IMG_EXTS = {'.jpg', '.jpeg', '.png'}
_CLASSES  = ['bumblebee', 'fly', 'butterfly', 'other', 'background', 'unsure']
_INS_CLS  = ['bumblebee', 'fly', 'butterfly', 'other']
_FIELD_SUBSETS = {
    'Set 1  (labeled_mb)':             ['labeled_mb'],
    'Set 2  (labeled_ls+labeled_set)': ['labeled_ls', 'labeled_set'],
}

def _count_dir(d):
    return sum(1 for f in d.iterdir() if f.suffix.lower() in _IMG_EXTS) if d.exists() else 0

print('\n' + '='*80)
print(f"{'Source':<34} {'BB':>5} {'Fly':>6} {'BM':>5} {'Other':>6} {'Insect':>7} {'BG':>7}")
print('-'*80)

_all = {c: 0 for c in _CLASSES}
for _label, _folders in _FIELD_SUBSETS.items():
    _c = {cls: sum(_count_dir(LABELED_DIR/fld/cls) for fld in _folders) for cls in _CLASSES}
    _ins = sum(_c[c] for c in _INS_CLS)
    print(f"{_label:<34} {_c['bumblebee']:>5} {_c['fly']:>6} {_c['butterfly']:>5} {_c['other']:>6} {_ins:>7} {_c['background']:>7}")
    for c in _CLASSES: _all[c] += _c[c]

_web_c = {cls: _count_dir(Path(WEB_DIR)/cls) for cls in _INS_CLS} if WEB_DIR and Path(WEB_DIR).exists() else {c:0 for c in _INS_CLS}
_web_ins = sum(_web_c.values())
print(f"{'Web (iNaturalist)':<34} {_web_c['bumblebee']:>5} {_web_c['fly']:>6} {_web_c['butterfly']:>5} {_web_c['other']:>6} {_web_ins:>7} {'':>7}")
print('-'*80)
_field_ins = sum(_all[c] for c in _INS_CLS)
print(f"{'Field total':<34} {_all['bumblebee']:>5} {_all['fly']:>6} {_all['butterfly']:>5} {_all['other']:>6} {_field_ins:>7} {_all['background']:>7}")
print(f"{'Grand total (field+web insect)':<34} {_all['bumblebee']+_web_c['bumblebee']:>5} {_all['fly']+_web_c['fly']:>6} {_all['butterfly']+_web_c['butterfly']:>5} {_all['other']+_web_c['other']:>6} {_field_ins+_web_ins:>7} {_all['background']:>7}")
print('='*80)
print('BB=bumblebee  BM=butterfly  BG=before quota sampling')
print('Web used in: binary (insect class) + group + 5-class classifiers')


In [ ]:
from collections import Counter
from pathlib import Path

def _source(p):
    p = Path(p)
    try:
        rel = p.relative_to(LABELED_DIR)
        return rel.parts[0]
    except ValueError:
        return 'web_images'

# ── Binary classifier ────────────────────────────────────────────
print('═'*55)
print('BINARY  (insect=1 / background=0)')
print('─'*55)
insect_src = Counter(_source(p) for p in insect_paths)
bg_src     = Counter(_source(p) for p in bg_sampled)
print('Insect paths by source:')
for k, v in sorted(insect_src.items()): print(f'  {k:<20} {v:>6}')
print(f'  {"TOTAL":<20} {sum(insect_src.values()):>6}')
print('Background paths (sampled) by source:')
for k, v in sorted(bg_src.items()):    print(f'  {k:<20} {v:>6}')
print(f'  {"TOTAL":<20} {sum(bg_src.values()):>6}')

# ── Group classifier ─────────────────────────────────────────────
print()
print('═'*55)
print('GROUP  (4-class: bumblebee/fly/butterfly/other)')
print('─'*55)
_cls_names = {i: c for i, c in enumerate(CLASSES_4)}
arc_by_cls = Counter(_cls_names[lbl] for _, lbl in arctic_smp)
web_by_cls = Counter(_cls_names[lbl] for _, lbl in web_smp)
print(f'  {"class":<14} {"field":>8} {"web":>8} {"total":>8}')
print('  ' + '-'*40)
for cls in CLASSES_4:
    a, w = arc_by_cls.get(cls, 0), web_by_cls.get(cls, 0)
    print(f'  {cls:<14} {a:>8} {w:>8} {a+w:>8}')
print(f'  {"TOTAL":<14} {sum(arc_by_cls.values()):>8} {sum(web_by_cls.values()):>8} {len(arctic_smp)+len(web_smp):>8}')


##### Cell 5 — Train binary classifier
Trains the insect vs background EfficientNet. Saves best weights to `models/binary_best.pth`.

In [ ]:
import shutil

def train_binary(backbone):
    print(f'\n{"#"*60}\n# Binary  {backbone.upper()}\n{"#"*60}')
    model = (build_efficientnet(2) if backbone == 'efficientnet'
             else build_insectnet(INSECTNET_W, 2)).to(DEVICE)
    tr_ldr = make_loader(binary_data, bin_tr, IMG_SIZE, BATCH, aug=True)
    va_ldr = make_loader(binary_data, bin_va, IMG_SIZE, BATCH, weighted=False)
    ckpt   = BINARY_DIR / f'binary_{backbone}_best.pth'
    model  = run_training(model, f'insect_detector_{backbone}',
                          tr_ldr, va_ldr,
                          EPOCHS_BINARY, LR_S1, bin_counts, ckpt,
                          DEVICE, CLASSES_BINARY, BINARY_DIR, IN_COLAB,
                          best_metric='recall_insect')
    # Test eval
    te_ldr = make_loader(binary_data, bin_te, IMG_SIZE, BATCH, weighted=False)
    te = eval_epoch(model, te_ldr,
                   __import__('torch').nn.CrossEntropyLoss(),
                   DEVICE, CLASSES_BINARY)
    ins_rec = te.get('per_recall', {}).get('insect', 0)
    print(f'Test: MacroF1={te["macro_f1"]:.3f}  InsectRecall={ins_rec:.3f}  Acc={te["acc"]:.3f}')
    if HAS_SKLEARN:
        print(classification_report(te['labels'], te['preds'],
                                    labels=list(range(len(CLASSES_BINARY))),
                                    target_names=CLASSES_BINARY, digits=3, zero_division=0))
    # Copy to models/ — inference notebooks look for binary_best.pth
    # If both backbones trained, the last one wins; use backbone-specific
    # name for archiving and also overwrite the generic name.
    shutil.copy(ckpt, MODEL_DIR / f'binary_{backbone}_best.pth')
    shutil.copy(ckpt, MODEL_DIR / 'binary_best.pth')  # used by inference
    print(f'  → models/binary_{backbone}_best.pth  +  binary_best.pth updated')
    if HAS_SKLEARN:
        rpt = classification_report(te['labels'], te['preds'],
                                    labels=list(range(len(CLASSES_BINARY))),
                                    target_names=CLASSES_BINARY, digits=3, zero_division=0)
        (BINARY_DIR / f'binary_{backbone}_test_report.txt').write_text(
            f'binary_{backbone}  test_macro_f1={te["macro_f1"]:.3f}'
            f'  insect_recall={ins_rec:.3f}\n\n' + rpt)
        print(f'  → binary_{backbone}_test_report.txt saved')

    if HAS_SKLEARN:
        import matplotlib.pyplot as _plt
        from sklearn.metrics import confusion_matrix as _cm
        _cm_arr = _cm(te['labels'], te['preds'])
        _fig, _ax = _plt.subplots(figsize=(5, 4))
        _im = _ax.imshow(_cm_arr, interpolation='nearest', cmap='Blues')
        _plt.colorbar(_im, ax=_ax)
        _ax.set_xticks(range(len(CLASSES_BINARY)))
        _ax.set_yticks(range(len(CLASSES_BINARY)))
        _ax.set_xticklabels([f'Pred: {c}' for c in CLASSES_BINARY], fontsize=10)
        _ax.set_yticklabels([f'True: {c}' for c in CLASSES_BINARY], fontsize=10)
        for _r in range(len(CLASSES_BINARY)):
            for _c in range(len(CLASSES_BINARY)):
                _color = 'white' if _cm_arr[_r, _c] > _cm_arr.max() / 2 else 'black'
                _ax.text(_c, _r, str(_cm_arr[_r, _c]), ha='center', va='center',
                         fontsize=13, fontweight='bold', color=_color)
        _ax.set_title(f'binary_{backbone} — Confusion Matrix (test set)', fontsize=11)
        _plt.tight_layout()
        _cm_path = BINARY_DIR / f'binary_{backbone}_confusion_matrix.png'
        _plt.savefig(_cm_path, dpi=120); _plt.close()
        print(f'  → binary_{backbone}_confusion_matrix.png saved')
    # Save structured results JSON
    import json as _json
    _ckpt_meta = __import__('torch').load(ckpt, map_location='cpu', weights_only=False)
    _results = {
        'model': f'binary_{backbone}',
        'img_size': IMG_SIZE,
        'best_epoch': int(_ckpt_meta.get('epoch', -1)),
        'best_val_f1': float(_ckpt_meta.get('val_score', 0)),
        'test_f1': float(te['macro_f1']),
        'test_recall': float(te.get('per_recall', {}).get('insect', 0)),
        'test_precision': float(te.get('per_precision', {}).get('insect', 0)),
        'test_acc': float(te['acc']),
        'test_tp': int(te.get('per_tp', {}).get('insect', 0)),
        'test_fp': int(te.get('per_fp', {}).get('insect', 0)),
        'test_fn': int(te.get('per_fn', {}).get('insect', 0)),
        'test_tn': int(te.get('per_tp', {}).get('background', 0)),
    }
    (BINARY_DIR / f'binary_{backbone}_results.json').write_text(_json.dumps(_results, indent=2))
    print(f'  → binary_{backbone}_results.json saved')
    _sync_to_drive(RUN_DIR, DRIVE_BASE, f'binary_{backbone}')
    # Sync updated model files to Drive/models/
    if IN_COLAB:
        import shutil as _sh_m
        _drive_models = __import__('pathlib').Path(DRIVE_BASE) / 'models'
        _drive_models.mkdir(parents=True, exist_ok=True)
        for _mname in [f'binary_{backbone}_best.pth', 'binary_best.pth']:
            _msrc = MODEL_DIR / _mname
            if _msrc.exists():
                _sh_m.copy2(str(_msrc), str(_drive_models / _mname))
                print(f'  ✓ Drive/models/{_mname} updated')
    return te

bin_res = {}
if BINARY_BACKBONE in ('efficientnet', 'both'): bin_res['efficientnet'] = train_binary('efficientnet')
if BINARY_BACKBONE in ('insectnet',    'both'): bin_res['insectnet']    = train_binary('insectnet')
if len(bin_res) > 1:
    print('\n=== BINARY COMPARISON ===')
    for name, r in bin_res.items():
        ins_rec = r.get('per_recall', {}).get('insect', 0)
        print(f'  {name:15}: F1={r["macro_f1"]:.3f}  InsectRecall={ins_rec:.3f}')


##### Cell 6 — Train group classifier
Trains the 4-class InsectNet (bumblebee / fly / butterfly / other). Saves to `models/4group_insectnet.pth`.

In [ ]:
import shutil

model_g = build_insectnet(INSECTNET_W, 4).to(DEVICE)

# Stage 1: web + arctic combined
if web_smp:
    combined = arctic_smp + web_smp
    comb_tr  = arc_tr + [len(arctic_smp)+i for i in web_tr]
    comb_cnt = {i: counts_arc.get(i,0)+counts_web.get(i,0) for i in range(4)}
    tr_s1 = make_loader(combined,  comb_tr, IMG_SIZE, BATCH, aug=True)
    va_s1 = make_loader(arctic_smp+web_smp,
                        arc_va+[len(arctic_smp)+i for i in web_va], IMG_SIZE, BATCH, weighted=False)
else:
    comb_cnt = counts_arc
    tr_s1 = make_loader(arctic_smp, arc_tr, IMG_SIZE, BATCH, aug=True)
    va_s1 = make_loader(arctic_smp, arc_va, IMG_SIZE, BATCH, weighted=False)

ckpt_s1 = GROUP_DIR/'4group_insectnet_stage1.pth'
model_g = run_training(model_g,'pollinator_classifier_s1',tr_s1,va_s1,
                       EPOCHS_S1,LR_S1,comb_cnt,ckpt_s1,DEVICE,CLASSES_4,GROUP_DIR,IN_COLAB)

# Stage 2: arctic fine-tune (skip by default)
ckpt_final = GROUP_DIR/'4group_insectnet.pth'
if EPOCHS_S2==0:
    print('Stage2=0 — copying Stage 1 as final model.')
    shutil.copy(ckpt_s1, ckpt_final)
else:
    model_g.load_state_dict(
        __import__('torch').load(ckpt_s1,map_location=DEVICE,weights_only=False)['state_dict'])
    tr_s2 = make_loader(arctic_smp, arc_tr, IMG_SIZE, BATCH, aug=True)
    va_s2 = make_loader(arctic_smp, arc_va, IMG_SIZE, BATCH, weighted=False)
    model_g = run_training(model_g,'pollinator_classifier_s2',tr_s2,va_s2,
                           EPOCHS_S2,LR_S2,counts_arc,ckpt_final,DEVICE,CLASSES_4,GROUP_DIR,IN_COLAB)

# Final test eval
model_g.load_state_dict(
    __import__('torch').load(ckpt_final,map_location=DEVICE,weights_only=False)['state_dict'])
te_arc = make_loader(arctic_smp, arc_te, IMG_SIZE, BATCH, weighted=False)
ta = eval_epoch(model_g,te_arc,
                __import__('torch').nn.CrossEntropyLoss(),DEVICE,CLASSES_4)
print(f'\nTest Arctic: MacroF1={ta["macro_f1"]:.3f}  Acc={ta["acc"]:.3f}')
if HAS_SKLEARN:
    print(classification_report(ta['labels'],ta['preds'],labels=list(range(len(CLASSES_4))),target_names=CLASSES_4,digits=3,zero_division=0))


# ── Test Web (iNaturalist) ───────────────────────────────────────
if web_smp and web_te:
    te_web = make_loader(web_smp, web_te, IMG_SIZE, BATCH, weighted=False)
    tw = eval_epoch(model_g, te_web,
                    __import__('torch').nn.CrossEntropyLoss(), DEVICE, CLASSES_4)
    print(f'Test Web   : MacroF1={tw["macro_f1"]:.3f}  Acc={tw["acc"]:.3f}  (iNaturalist generalisation)')
    if HAS_SKLEARN:
        print(classification_report(tw['labels'], tw['preds'], labels=list(range(len(CLASSES_4))), target_names=CLASSES_4, digits=3, zero_division=0))
else:
    tw = None
    print('No web test set available.')
# Copy best checkpoint to models/ so inference notebooks stay unchanged
shutil.copy(ckpt_final, MODEL_DIR / '4group_insectnet.pth')
print(f'  → models/4group_insectnet.pth updated')

# Save classification report
if HAS_SKLEARN:
    rpt = classification_report(ta['labels'], ta['preds'],
                                labels=list(range(len(CLASSES_4))),
                                target_names=CLASSES_4, digits=3, zero_division=0)
    web_rpt = (classification_report(tw['labels'], tw['preds'],
               labels=list(range(len(CLASSES_4))), target_names=CLASSES_4,
               digits=3, zero_division=0) if tw else 'No web test data.')
    (GROUP_DIR / 'group4_test_report.txt').write_text(
        f'4group_insectnet  test_arctic_macro_f1={ta["macro_f1"]:.3f}\n\n'
        f'-- Test Arctic (field crops) --\n' + rpt +
        f'\n-- Test Web (iNaturalist) --\n' + web_rpt)
    print('  → group4_test_report.txt saved')


if HAS_SKLEARN:
    import matplotlib.pyplot as _plt
    from sklearn.metrics import confusion_matrix as _cm
    _cm_arr = _cm(ta['labels'], ta['preds'])
    _fig, _ax = _plt.subplots(figsize=(6, 5))
    _im = _ax.imshow(_cm_arr, interpolation='nearest', cmap='Blues')
    _plt.colorbar(_im, ax=_ax)
    _ax.set_xticks(range(len(CLASSES_4)))
    _ax.set_yticks(range(len(CLASSES_4)))
    _ax.set_xticklabels([f'Pred: {c}' for c in CLASSES_4], fontsize=9, rotation=20, ha='right')
    _ax.set_yticklabels([f'True: {c}' for c in CLASSES_4], fontsize=9)
    for _r in range(len(CLASSES_4)):
        for _c in range(len(CLASSES_4)):
            _color = 'white' if _cm_arr[_r, _c] > _cm_arr.max() / 2 else 'black'
            _ax.text(_c, _r, str(_cm_arr[_r, _c]), ha='center', va='center',
                     fontsize=12, fontweight='bold', color=_color)
    _ax.set_title('4group_insectnet — Confusion Matrix (test set)', fontsize=11)
    _plt.tight_layout()
    _cm_path = GROUP_DIR / 'group4_confusion_matrix.png'
    _plt.savefig(_cm_path, dpi=120); _plt.close()
    print('  → group4_confusion_matrix.png saved')
# Save structured results JSON
import json as _json
_ckpt_meta = __import__('torch').load(ckpt_final, map_location='cpu', weights_only=False)
_results_g = {
    'model': '4group_insectnet',
    'img_size': IMG_SIZE,
    'classes': CLASSES_4,
    'best_epoch': int(_ckpt_meta.get('epoch', -1)),
    'val_macro_f1': float(_ckpt_meta.get('val_macro_f1', 0)),
    'val_acc': float(_ckpt_meta.get('val_acc', 0)),
    'val_per_class_f1': {k: float(v) for k, v in _ckpt_meta.get('val_per_class_f1', {}).items()},
    'test_arctic_macro_f1': float(ta['macro_f1']),
    'test_arctic_acc': float(ta['acc']),
    'test_arctic_per_class': {k: float(v) for k, v in ta.get('per_f1', {}).items()},
    'test_web_macro_f1': float(tw['macro_f1']) if tw else None,
    'test_web_acc': float(tw['acc']) if tw else None,
    'test_web_per_class': {k: float(v) for k, v in tw.get('per_f1', {}).items()} if tw else {},
}
(GROUP_DIR / 'group4_results.json').write_text(_json.dumps(_results_g, indent=2))
print('  → group4_results.json saved')

_sync_to_drive(RUN_DIR, DRIVE_BASE, '4group_insectnet')
# Sync 4group model to Drive/models/
if IN_COLAB:
    import shutil as _sh_m
    _drive_models = __import__('pathlib').Path(DRIVE_BASE) / 'models'
    _drive_models.mkdir(parents=True, exist_ok=True)
    _msrc = MODEL_DIR / '4group_insectnet.pth'
    if _msrc.exists():
        _sh_m.copy2(str(_msrc), str(_drive_models / '4group_insectnet.pth'))
        print('  ✓ Drive/models/4group_insectnet.pth updated')
print(f'\nAll outputs in: {RUN_DIR}')
